In [1]:
!nvidia-smi

Sat May 24 21:38:25 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       On  |   00000000:00:1E.0 Off |                    0 |
| N/A   26C    P8              8W /   70W |       1MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install --upgrade openai pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 720.4/720.4 kB 51.4 MB/s eta 0:00:00


In [3]:
import openai                           
import base64                          
from PIL import Image
import json

In [ ]:
openai.api_key = ""

In [5]:
def encode_image(image_path):
    """
    Encode an image to base64 format for GPT-4 Vision input.
    """
    try:
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode("utf-8")
    except FileNotFoundError:
        print(f"Error: Image file not found at {image_path}")
        return None
    except Exception as e:
        print(f"Error encoding image: {e}")
        return None

In [6]:
def extract_palm_features(image_base64, yolo_json_data):
    """
    Analyzes a palm image and YOLOv11 JSON data to extract palm line features using GPT-4o.
    """
    if not image_base64:
        return "Error: Image data is missing."

    prompt = f"""
    You are an expert palmistry analyst. Analyze the hand image and YOLOv11 JSON data using these **metric-driven protocols**:
    
    **Operational Definitions**
    1. **Length**: (arc_length/max_palm_width)×100 → short(<40%), medium(40-75%), long(>75%)
    2. **Depth**: deep if stroke_width ≤1.5×background_crease, else shallow
    3. **Branches**: none/upward/downward/mixed (splits ≥15% length at <45°)
    4. **Forks**: true if >25% width separation at distal segment
    5. **Breaks**: true if discontinuity >5% palm width
    6. **Coordinates**: Normalized to [0-1] scale relative to palm bounding box
    
    JSON data:
    ```json
    {json.dumps(yolo_json_data, indent=2)}
    
    Required Features per Line
    
    length: short/medium/long/not_determinable (calculated via palm width %)
    
    depth: deep/shallow/not_determinable (stroke width ratio)
    
    clarity: clear/chained/blurry/not_determinable
    
    curvature: straight(<5°)/curved/downward/irregular/not_determinable
    
    branches: none/upward/downward/mixed (categorical)
    
    forks: true/false (major fork)
    
    breaks: true/false (significant discontinuity)
    
    islands: true/false (closed loops override forks)
    
    start_mount: Jupiter/Saturn/Apollo/Mercury/Venus/Moon/unknown
    
    end_mount: Jupiter/Saturn/Apollo/Mercury/Venus/Moon/unknown
    
    start_xy: [x,y] (normalized 0-1 coordinates)
    
    end_xy: [x,y] (normalized 0-1 coordinates)
    
    Output Format 
    
    {{
      "hand": "right/left/unknown",
      "lines": [
        {{
          "line_type": "heart-line", 
          "yolo_confidence": 0.90569,
          "yolo_box": {{ "x1": 609.73, "y1": 702.86, "x2": 911.74, "y2": 848.86 }},
          "features": {{
            "length": "medium",
            "depth": "deep",
            "clarity": "chained",
            "curvature": "straight",
            "branches": "upward",  
            "forks": false,
            "breaks": false,
            "islands": true,
            "start_mount": "Jupiter",
            "end_mount": "Mercury",
            "start_xy": [0.18, 0.12],
            "end_xy": [0.78, 0.14]
          }}
        }}
      ],
      "variant": "simian",  
      "quality_issues": ["low_confidence"]  // Confidence <0.6
    }}
    
    Analyze the image carefully. Use the bounding box information from the JSON to focus on the correct part of the image for each line. Be as detailed and accurate as possible. If a line mentioned in the JSON is not clearly visible or its features cannot be determined from the image, use "not determinable" for the descriptive string features. For 'branches', 'forks', 'breaks', and 'islands', if their presence cannot be determined due to poor visibility of the line segment, then outputting false is acceptable, but prioritize detection if the line segment is clear.
    """
    
    try:
        response = openai.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are an expert palmistry analyst."},
                {"role": "user", "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}}
                ]}
            ],
            max_tokens=2000, 
            temperature=0.2 
        )
        if not response.choices or not response.choices[0].message or not response.choices[0].message.content:
            print("Warning: No content returned from GPT-4o.")
            return "Error: No content returned from GPT-4o."
        
        # The response content should be a JSON string, so we try to parse it
        response_content = response.choices[0].message.content.strip()
        
        # GPT might return the JSON block within triple backticks, remove them if present
        if response_content.startswith("```json"):
            response_content = response_content[7:]
        if response_content.endswith("```"):
            response_content = response_content[:-3]
            
        return json.loads(response_content) # Parse the JSON string into a Python dictionary

    except json.JSONDecodeError as e:
        print(f"Error decoding JSON response from GPT-4o: {e}")
        print(f"Raw response: {response_content}")
        return f"Error: Could not parse JSON response. Raw response: {response_content}"
    except Exception as e:
        print(f"An error occurred while calling OpenAI API: {e}")
        return f"Error: An API error occurred: {e}"

In [7]:
image_path = "3_output.jpg"

In [8]:
try:
    with open(image_path, "rb") as f:
        pass
except FileNotFoundError:
    print(f"Warning: The image '{image_path}' was not found.")

In [9]:
b64_image_data = encode_image(image_path)

In [10]:
yolo_output_data_str = """
[
    {
        "name": "Head-Line",
        "class": 1,
        "confidence": 0.90179,
        "box": {
            "x1": 602.77264,
            "y1": 589.64612,
            "x2": 899.05829,
            "y2": 775.95465
        }
    },
    {
        "name": "Heart-Line",
        "class": 2,
        "confidence": 0.89062,
        "box": {
            "x1": 494.44461,
            "y1": 604.86993,
            "x2": 736.64441,
            "y2": 725.33051
        }
    },
    {
        "name": "Life-Line",
        "class": 3,
        "confidence": 0.88568,
        "box": {
            "x1": 665.4881,
            "y1": 598.60773,
            "x2": 902.50354,
            "y2": 949.0119
        }
    },
    {
        "name": "Fate-Line",
        "class": 0,
        "confidence": 0.30095,
        "box": {
            "x1": 625.69238,
            "y1": 688.49951,
            "x2": 702.7843,
            "y2": 838.08582
        }
    }
]
"""

yolo_json_data = json.loads(yolo_output_data_str)

In [11]:
filtered_yolo_data = [item for item in yolo_json_data if item and "name" in item]

In [12]:
extracted_features = extract_palm_features(b64_image_data, filtered_yolo_data)

print("--- Extracted Palm Features (JSON Output) ---")
if isinstance(extracted_features, dict): # Check if it's already a dictionary
    print(json.dumps(extracted_features, indent=2))
else: # If it's a string
    print(extracted_features)

--- Extracted Palm Features (JSON Output) ---
{
  "hand": "right",
  "lines": [
    {
      "line_type": "head-line",
      "yolo_confidence": 0.90179,
      "yolo_box": {
        "x1": 602.77,
        "y1": 589.65,
        "x2": 899.06,
        "y2": 775.95
      },
      "features": {
        "length": "medium",
        "depth": "deep",
        "clarity": "clear",
        "curvature": "straight",
        "branches": "none",
        "forks": false,
        "breaks": false,
        "islands": false,
        "start_mount": "Jupiter",
        "end_mount": "Mercury",
        "start_xy": [
          0.6,
          0.58
        ],
        "end_xy": [
          0.9,
          0.77
        ]
      }
    },
    {
      "line_type": "heart-line",
      "yolo_confidence": 0.89062,
      "yolo_box": {
        "x1": 494.44,
        "y1": 604.87,
        "x2": 736.64,
        "y2": 725.33
      },
      "features": {
        "length": "medium",
        "depth": "deep",
        "clarity": "clear",
 